In [ ]:
#STAGE 1 - CONFOUNDER REGRESSION

"""
Residualises GFAP and NfL against age, sex, BMI, eGFR using the full baseline
cohort. Saves the adjusted biomarkers as a CSV for downstream group-stratified
cognitive analysis in Script 2.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats
import os

biomarkers = ['gfap', 'nefl']

output_dir = 'stage1_confound/'
os.makedirs(output_dir, exist_ok=True)

data = pd.read_csv(
    'cog_biomarker_merged.csv',
    usecols=['eid', 'Sex', 'test_name', 'instance', 'age_at_test', 'z_score',
             'BMI', 'eGFR', 'gfap', 'nefl', 'hc', 'ic']
)
data = data.dropna(subset=['z_score', 'BMI', 'eGFR', 'gfap', 'nefl'])
data_0 = data[data['instance'] == 0].copy()
tests = data['test_name'].unique()
test_direction = {
    'PMat': -1, 'RT': -1, 'FI': 1, 'NM': 1, 'SD': 1,
    'TMAT': -1, 'TMB': -1, 'MP': 1, 'PAL': 1, 'PV': 1, 'TR': 1, 'BL': 1
}
data_0['z_adj'] = data_0['z_score'] * data_0['test_name'].map(test_direction)

In [ ]:
#STAGE 1 - ADJUST BIOMARKERS FOR CONFOUNDERS

"""
Residualises each biomarker against age, sex, BMI, eGFR using OLS.
Residuals = biomarker variation NOT explained by confounders, used as the
adjusted biomarker downstream in Stage 2.
Coefficient tables, fit statistics, and model summary text are saved per biomarker.
"""

def format_p(p):
    return f"{p:.2e}" if p < 0.001 else f"{p:.4f}"

for biomarker in biomarkers:

    print(f"\nAdjusting {biomarker} for confounders")

    # Fit confounder regression
    adjustment_formula = f'{biomarker} ~ age_at_test + BMI + eGFR + C(Sex)'
    adjustment_model = smf.ols(adjustment_formula, data=data_0).fit()
    print(adjustment_model.summary())

    # Assign residuals back to data_0 by index (handles any rows statsmodels dropped)
    adjusted_col = f'{biomarker}_adjusted'
    data_0[adjusted_col] = np.nan
    data_0.loc[adjustment_model.resid.index, adjusted_col] = adjustment_model.resid

    print(f"  Residuals assigned: {data_0[adjusted_col].notna().sum()} / {len(data_0)} rows")

    # Verify adjustment worked - correlations should be ~0
    residuals = data_0[adjusted_col].dropna()
    aligned = data_0.loc[residuals.index]

    print("\n  Verifying adjustment:")
    print(f"    {adjusted_col} vs Age:  {np.corrcoef(residuals, aligned['age_at_test'])[0,1]:.6f}")
    print(f"    {adjusted_col} vs BMI:  {np.corrcoef(residuals, aligned['BMI'])[0,1]:.6f}")
    print(f"    {adjusted_col} vs eGFR: {np.corrcoef(residuals, aligned['eGFR'])[0,1]:.6f}")

    # Save coefficient table
    adj_coef_table = pd.DataFrame({
        'biomarker': biomarker,
        'term': adjustment_model.params.index,
        'estimate': adjustment_model.params.values,
        'std_error': adjustment_model.bse.values,
        't_statistic': adjustment_model.tvalues.values,
        'p_value': adjustment_model.pvalues.values,
        'ci_low': adjustment_model.conf_int()[0].values,
        'ci_high': adjustment_model.conf_int()[1].values
    })

    adj_dir = os.path.join(output_dir, 'adjustment_models')
    os.makedirs(adj_dir, exist_ok=True)

    with open(os.path.join(adj_dir, f'{biomarker}_adjustment_summary.txt'), 'w') as f:
        f.write(adjustment_model.summary().as_text())

    fit_stats = pd.DataFrame({
        'biomarker': biomarker,
        'metric': ['R²', 'Adjusted R²', 'AIC', 'BIC', 'F-statistic', 'F p-value',
                   'Log-Likelihood', 'Df Residuals', 'Df Model', 'N'],
        'value': [adjustment_model.rsquared,
                  adjustment_model.rsquared_adj,
                  adjustment_model.aic,
                  adjustment_model.bic,
                  adjustment_model.fvalue,
                  format_p(adjustment_model.f_pvalue),
                  adjustment_model.llf,
                  adjustment_model.df_resid,
                  adjustment_model.df_model,
                  adjustment_model.nobs]
    })
    fit_stats.to_csv(os.path.join(adj_dir, f'{biomarker}_adjustment_fit_stats.csv'), index=False)
    adj_coef_table.to_csv(os.path.join(adj_dir, f'{biomarker}_adjustment_coefficients.csv'), index=False)

print("\nBiomarkers adjusted for age, sex, BMI, eGFR")
print(f"Columns now in data_0: {[c for c in data_0.columns if 'adjusted' in c]}")

In [ ]:
# CELL 2: VISUALIZE ADJUSTMENT SUCCESS
"""
Creates plots showing that adjustment removed confounder effects.
Before adjustment: biomarker correlates with age/eGFR
After adjustment: biomarker does not correlate with age/eGFR

LAYOUT (4 rows × 2 columns):
Row 1: Age (before | after)
Row 2: eGFR (before | after)  
Row 3: BMI (before | after)
Row 4: Sex (before | after) ← BOXPLOTS for categorical variable
"""

from scipy import stats

for biomarker in biomarkers:
    
    # Get complete data for plotting
    required_vars = [biomarker, f'{biomarker}_adjusted', 'age_at_test', 'eGFR', 'BMI', 'Sex']
    plot_df = data_0[required_vars].dropna()
    
    # Create 2x4 plot
    fig, axes = plt.subplots(4, 2, figsize=(14, 16))
    fig.suptitle(f'{biomarker.upper()} Adjustment Verification', 
                 fontsize=16, fontweight='bold')
    
    #PLOT 1: Original biomarker vs age
    axes[0, 0].scatter(plot_df['age_at_test'], plot_df[biomarker], 
                       alpha=0.05, s=20, c='darkgrey', edgecolors='none')
    axes[0, 0].set_xlabel('Age at Test', fontsize=11)
    axes[0, 0].set_ylabel(f'Original {biomarker.upper()} (NPX)', fontsize=11)
    axes[0, 0].set_title('Before Adjustment: Age Effect Present', fontsize=12)

    #create regression line
    slope, intercept,  r_value, p_value, std_err = stats.linregress(
        plot_df['age_at_test'],
        plot_df[biomarker])
    line_x = np.linspace(plot_df['age_at_test'].min(),
                         plot_df['age_at_test'].max(),
                         100)
    line_y = slope*line_x+intercept

    #plot the line
    axes[0,0].plot(line_x, line_y, 
                   'k-', #line colour
                   linewidth=1, 
                   label=f'Slope={slope:.3f}') #shows legend 
    
    # Add correlation text
    corr_orig = np.corrcoef(plot_df['age_at_test'], plot_df[biomarker])[0,1]
    axes[0, 0].text(0.05, 0.95, f'r = {corr_orig:.3f}', 
                    transform=axes[0, 0].transAxes, 
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    #PLOT 2: Adjusted biomarker vs age (should be flat)
    axes[0, 1].scatter(plot_df['age_at_test'], plot_df[f'{biomarker}_adjusted'], 
                       alpha=0.05, s=20, c='darkgrey', edgecolors='none')
    axes[0, 1].set_xlabel('Age at Test', fontsize=11)
    axes[0, 1].set_ylabel(f'Adjusted {biomarker.upper()} (residuals)', fontsize=11)
    axes[0, 1].set_title('After Adjustment: Age Effect Removed', fontsize=12)

    #create regression line
    slope_adj, intercept_adj,  r_value_adj, p_value_adj, std_err_adj = stats.linregress(
        plot_df['age_at_test'],
        plot_df[f'{biomarker}_adjusted'])
    line_x_adj = np.linspace(plot_df['age_at_test'].min(),
                         plot_df['age_at_test'].max(),
                         100)
    line_y_adj = slope_adj*line_x_adj+intercept_adj

    #plot the line
    axes[0,1].plot(line_x_adj, line_y_adj, 
                   'k-', #line colour
                   linewidth=1, 
                   label=f'Slope={slope_adj:.3f}') #shows legend 
    
    # Add correlation text
    corr_adj = np.corrcoef(plot_df['age_at_test'], 
                           plot_df[f'{biomarker}_adjusted'])[0,1]
    axes[0, 1].text(0.05, 0.95, f'r = {corr_adj:.3f}', 
                    transform=axes[0, 1].transAxes, 
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    #PLOT 3: Original biomarker vs eGFR
    axes[1, 0].scatter(plot_df['eGFR'], plot_df[biomarker], 
                       alpha=0.05, s=20, c='darkgrey', edgecolors='none')
    axes[1, 0].set_xlabel('eGFR (mL/min/1.73m²)', fontsize=11)
    axes[1, 0].set_ylabel(f'Original {biomarker.upper()} (NPX)', fontsize=11)
    axes[1, 0].set_title('Before Adjustment: eGFR Effect Present', fontsize=12)
    
    # Add correlation text
    corr_orig_egfr = np.corrcoef(plot_df['eGFR'], plot_df[biomarker])[0,1]
    axes[1, 0].text(0.05, 0.95, f'r = {corr_orig_egfr:.3f}', 
                    transform=axes[1, 0].transAxes, 
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    #create regression line
    slope, intercept,  r_value, p_value, std_err = stats.linregress(
        plot_df['eGFR'],
        plot_df[biomarker])
    line_x = np.linspace(plot_df['eGFR'].min(),
                         plot_df['eGFR'].max(),
                         100)
    line_y = slope*line_x+intercept

    #plot the line
    axes[1,0].plot(line_x, line_y, 
                   'k-', #line colour
                   linewidth=1, 
                   label=f'Slope={slope:.3f}') #shows legend 
    
    #PLOT 4: Adjusted biomarker vs eGFR (should be flat)
    axes[1, 1].scatter(plot_df['eGFR'], plot_df[f'{biomarker}_adjusted'], 
                       alpha=0.05, s=20, c='darkgrey', edgecolors='none')
    axes[1, 1].set_xlabel('eGFR (mL/min/1.73m²)', fontsize=11)
    axes[1, 1].set_ylabel(f'Adjusted {biomarker.upper()} (residuals)', fontsize=11)
    axes[1, 1].set_title('After Adjustment: eGFR Effect Removed', fontsize=12)

    #create regression line
    slope_adj, intercept_adj,  r_value_adj, p_value_adj, std_err_adj = stats.linregress(
        plot_df['eGFR'],
        plot_df[f'{biomarker}_adjusted'])
    line_x_adj = np.linspace(plot_df['eGFR'].min(),
                         plot_df['eGFR'].max(),
                         100)
    line_y_adj = slope_adj*line_x_adj+intercept_adj
    
    #plot the line
    axes[1,1].plot(line_x_adj, line_y_adj, 
                   'k-', #line colour
                   linewidth=1, 
                   label=f'Slope={slope:.3f}') #shows legend 
    
    
    # Add correlation text
    corr_adj_egfr = np.corrcoef(plot_df['eGFR'], 
                                plot_df[f'{biomarker}_adjusted'])[0,1]
    axes[1, 1].text(0.05, 0.95, f'r = {corr_adj_egfr:.3f}', 
                    transform=axes[1, 1].transAxes, 
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    #PLOT 5: Original biomarker vs BMI
    axes[2, 0].scatter(plot_df['BMI'], plot_df[biomarker], 
                       alpha=0.05, s=20, c='darkgrey', edgecolors='none')
    axes[2, 0].set_xlabel('BMI', fontsize=11)
    axes[2, 0].set_ylabel(f'Original {biomarker.upper()} (NPX)', fontsize=11)
    axes[2, 0].set_title('Before Adjustment: BMI Effect Present', fontsize=12)

    #create regression line
    slope, intercept,  r_value, p_value, std_err = stats.linregress(
        plot_df['BMI'],
        plot_df[biomarker])
    line_x = np.linspace(plot_df['BMI'].min(),
                         plot_df['BMI'].max(),
                         100)
    line_y = slope*line_x+intercept

    #plot the line
    axes[2,0].plot(line_x, line_y, 
                   'k-', #line colour
                   linewidth=1, 
                   label=f'Slope={slope:.3f}') #shows legend 
    
    # Add correlation text
    corr_orig = np.corrcoef(plot_df['BMI'], plot_df[biomarker])[0,1]
    axes[2, 0].text(0.05, 0.95, f'r = {corr_orig:.3f}', 
                    transform=axes[2, 0].transAxes, 
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    #PLOT 6: Adjusted biomarker vs BMI (should be flat)
    axes[2, 1].scatter(plot_df['BMI'], plot_df[f'{biomarker}_adjusted'], 
                       alpha=0.05, s=20, c='darkgrey', edgecolors='none')
    axes[2, 1].set_xlabel('BMI', fontsize=11)
    axes[2, 1].set_ylabel(f'Adjusted {biomarker.upper()} (residuals)', fontsize=11)
    axes[2, 1].set_title('After Adjustment: BMI Effect Removed', fontsize=12)

    #create regression line
    slope_adj, intercept_adj,  r_value_adj, p_value_adj, std_err_adj = stats.linregress(
        plot_df['BMI'],
        plot_df[f'{biomarker}_adjusted'])
    line_x_adj = np.linspace(plot_df['BMI'].min(),
                         plot_df['BMI'].max(),
                         100)
    line_y_adj = slope_adj*line_x_adj+intercept_adj

    #plot the line
    axes[2,1].plot(line_x_adj, line_y_adj, 
                   'k-', #line colour
                   linewidth=1, 
                   label=f'Slope={slope:.3f}') #shows legend 
    
    # Add correlation text
    corr_adj = np.corrcoef(plot_df['BMI'], 
                           plot_df[f'{biomarker}_adjusted'])[0,1]
    axes[2, 1].text(0.05, 0.95, f'r = {corr_adj:.3f}', 
                    transform=axes[2, 1].transAxes, 
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    #PLOT 7: Original biomarker vs sex
    # Create boxplot using seaborn (easier than matplotlib for this)
    sns.boxplot(x='Sex', y=biomarker, data=plot_df, 
               ax=axes[3, 0],
               color='lightgrey',  # Box color
               width=0.5, # Box width
               flierprops={
               'marker': 'o',           # Circle marker
               'markerfacecolor': 'darkgrey',  # Fill color
               'markersize': 4,         # Size
               'markeredgecolor': 'none',      # ← Remove edge!
               'alpha': 0.05             # Transparency
               })  
    
    axes[3, 0].set_xlabel('Sex', fontsize=11)
    axes[3, 0].set_ylabel(f'Original {biomarker.upper()} (NPX)', fontsize=11)
    axes[3, 0].set_title('Before Adjustment: Sex Effect Present', fontsize=12)
    
    # Calculate mean difference between groups
    # .groupby() splits data by Sex, then calculates mean for each group
    # .diff() calculates difference between consecutive values
    # .iloc[-1] gets the last value (the difference)
    mean_male = plot_df[plot_df['Sex'] == 0][biomarker].mean()
    mean_female = plot_df[plot_df['Sex'] == 1][biomarker].mean()
    mean_diff = mean_female - mean_male  # Difference: Female - Male
    
    # Add text showing the difference
    axes[3, 0].text(0.05, 0.95, 
                   f'Female - Male\nΔ = {mean_diff:.3f}', 
                   transform=axes[3, 0].transAxes, 
                   verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    #PLOT 8: Adjusted biomaker vs Sex
    sns.boxplot(x='Sex', y=f'{biomarker}_adjusted', data=plot_df,
               ax=axes[3, 1],
               color='lightgrey',
               width=0.5,
               flierprops={
               'marker': 'o',           # Circle marker
               'markerfacecolor': 'darkgrey',  # Fill color
               'markersize': 4,         # Size
               'markeredgecolor': 'none',      # ← Remove edge!
               'alpha': 0.05             # Transparency
               })
    
    axes[3, 1].set_xlabel('Sex', fontsize=11)
    axes[3, 1].set_ylabel(f'Adjusted {biomarker.upper()} (residuals)', fontsize=11)
    axes[3, 1].set_title('After Adjustment: Sex Effect Removed', fontsize=12)
    
    # Calculate mean difference (should be ~0 after adjustment)
    mean_male_adj = plot_df[plot_df['Sex'] == 0][f'{biomarker}_adjusted'].mean()
    mean_female_adj = plot_df[plot_df['Sex'] == 1][f'{biomarker}_adjusted'].mean()
    mean_diff_adj = mean_female_adj - mean_male_adj
    
    # Add text showing difference is now ~0
    axes[3, 1].text(0.05, 0.95, 
                   f'Female - Male\nΔ = {mean_diff_adj:.3f}', 
                   transform=axes[3, 1].transAxes, 
                   verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    
    # Save figure
    fig_dir = os.path.join(output_dir, 'adjustment_verification')
    os.makedirs(fig_dir, exist_ok=True)
    plt.savefig(os.path.join(fig_dir, f'{biomarker}_adjustment_check.png'), 
                dpi=800, bbox_inches='tight')
    plt.show()
    
    print(f"Saved adjustment verification plot for {biomarker}")

In [ ]:
#SAVE ADJUSTED DATAFRAME FOR STAGE 2

"""
Writes the dataframe containing adjusted biomarker residuals, group flags,
and z_adj to disk. Script 2 reads this file directly, avoiding any need
to re-fit the Stage 1 models.
"""

save_cols = ['eid', 'Sex', 'test_name', 'instance', 'age_at_test', 'z_score',
             'z_adj', 'BMI', 'eGFR', 'gfap', 'nefl',
             'gfap_adjusted', 'nefl_adjusted', 'hc', 'ic']

data_0[save_cols].to_csv(
    os.path.join(output_dir, 'stage1_adjusted_data.csv'),
    index=False
)

print(f"Saved adjusted dataframe: {len(data_0)} rows, "
      f"{data_0['eid'].nunique()} unique people")
print(f"  HC observations: {(data_0['hc'] == 1).sum()}")
print(f"  IC observations: {(data_0['ic'] == 1).sum()}")
print(f"  Unclassified observations: {((data_0['hc'] == 0) & (data_0['ic'] == 0)).sum()}")